# Week 12 실습 · 탐색적 데이터 분석과 프로젝트 기획

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week12/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. **질문 → 데이터 이해 → 단변량 → 이변량·그룹 비교 → 발견 정리**의 EDA 절차를 따라 분석합니다.
2. 날짜·시각에서 **파생 변수**(월, 요일, 시간대)를 만들고, **집계 단위**(주문 건수·판매 수량·매출)를 구분합니다.
3. 그룹마다 일수가 다를 때 **합계가 아닌 일평균**으로 공정하게 비교합니다.
4. 두 데이터(카페 판매 + 기온)를 날짜로 합쳐 관계를 살펴보고, **상관과 인과**를 구분합니다.
5. 최종 프로젝트의 **연구 질문**과 **제안서**를 작성합니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | EDA 절차와 데이터 이해 | 15분 |
| 2 | Level 1 파생 변수 만들기 | 15분 |
| 3 | Level 1~2 단변량 · 그룹 비교 | 25분 |
| 4 | Level 3 두 데이터 합치기: 기온과 판매 | 20분 |
| 5 | Level 4 AI 분석 해석 검증 | 15분 |
| 6 | 최종 프로젝트 기획 | 25분 |
| 7 | 정리 | 5분 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request

try:
    import koreanize_matplotlib            # 그래프 한글 폰트
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

cafe = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])
weather = pd.read_csv(get_data("weather_daily.csv"), parse_dates=["date"])
print(cafe.shape, weather.shape)

---
## 1. EDA 절차와 데이터 이해

| 단계 | 질문 | 도구 |
|---|---|---|
| ① 질문 | 무엇을 알고 싶은가? | 연구 질문 목록 |
| ② 데이터 이해 | 한 행은 무엇인가? 기간·단위·결측은? | `info`, `describe`, 기간 |
| ③ 단변량 | 각 변수는 어떻게 분포하나? | `value_counts`, `describe`, 히스토그램 |
| ④ 이변량·그룹 | 변수끼리, 그룹끼리 어떻게 다른가? | `groupby`, `pivot_table`, `corr` |
| ⑤ 발견 정리 | 숫자와 함께, 한계와 함께 | 발견 3개 + 주의점 |

**오늘의 질문 (캠퍼스 카페 사장님의 궁금증)**

- Q1. 어떤 메뉴가 매출을 이끄는가?
- Q2. 언제(요일·시간대) 가장 바쁜가?
- Q3. 시험 기간에 판매가 달라지는가?
- Q4. 날씨(기온)는 판매와 관련이 있는가?

In [ ]:
cafe.head()

In [ ]:
cafe.info()
print("기간:", cafe["date"].min().date(), "~", cafe["date"].max().date(), "/ 영업일 수:", cafe["date"].nunique())
print("결측:", cafe.isna().sum().sum())

> **한 행 = 주문 한 건의 메뉴 한 종류** (같은 메뉴 여러 잔이면 `qty`가 2 이상). 따라서
> - **주문 건수** = 행 수
> - **판매 수량** = `qty`의 합
> - **매출** = `unit_price × qty`의 합
>
> 셋은 서로 다른 질문의 답입니다. "인기 메뉴"를 무엇으로 볼지 먼저 정해야 합니다.

---
## 2. Level 1 따라하기: 파생 변수 만들기

`.dt` 접근자로 날짜에서 월·요일·주차 등을 꺼냅니다.

In [ ]:
WEEKDAYS = ["월", "화", "수", "목", "금", "토", "일"]

cafe["revenue"] = cafe["unit_price"] * cafe["qty"]
cafe["month"] = cafe["date"].dt.month
cafe["weekday"] = pd.Categorical(cafe["date"].dt.dayofweek.map(dict(enumerate(WEEKDAYS))),
                                 categories=WEEKDAYS[:6], ordered=True)     # 일요일은 휴무
cafe["hour"] = cafe["time"].str[:2].astype(int)
cafe["daypart"] = pd.cut(cafe["hour"], bins=[7, 10, 13, 16, 19],
                         labels=["아침", "점심", "오후", "저녁"])        # (7,10] = 8~10시 ...

EXAMS = pd.to_datetime(["2025-04-21", "2025-06-16"])                   # 중간·기말고사 주 월요일
cafe["exam_week"] = cafe["date"].apply(lambda d: any(0 <= (d - s).days < 5 for s in EXAMS))

cafe[["date", "time", "menu", "revenue", "month", "weekday", "hour", "daypart", "exam_week"]].head()

> 💡 `pd.Categorical(..., ordered=True)`로 요일 순서를 정해 두면 정렬·그래프에서 **월화수…** 순서가 유지됩니다. (그냥 문자열이면 가나다순 → 금목수월토화)

---
## 3. 단변량 · 그룹 비교

### Q1. 어떤 메뉴가 매출을 이끄는가? – 세 가지 기준 비교

In [ ]:
menu = cafe.groupby("menu").agg(orders=("order_id", "size"), qty=("qty", "sum"), revenue=("revenue", "sum"))
menu["share"] = (menu["revenue"] / menu["revenue"].sum() * 100).round(1)
menu = menu.sort_values("revenue", ascending=False)
menu["rank_orders"] = menu["orders"].rank(ascending=False).astype(int)
menu["rank_revenue"] = range(1, len(menu) + 1)
menu

In [ ]:
ax = menu["share"].plot.barh(figsize=(6, 4), color="#4C78A8")
ax.invert_yaxis()
ax.set_xlabel("매출 비중 (%)")
ax.set_title("메뉴별 매출 비중")
plt.tight_layout()
plt.show()

### Q2. 언제 가장 바쁜가? – 합계가 아닌 **일평균**

요일마다 영업일 수가 다를 수 있습니다(공휴일, 기간의 시작·끝). 그래서 **요일별 합계 ÷ 그 요일의 영업일 수**로 비교합니다.

In [ ]:
daily = cafe.groupby("date").agg(revenue=("revenue", "sum"), orders=("order_id", "size")).reset_index()
daily["weekday"] = pd.Categorical(daily["date"].dt.dayofweek.map(dict(enumerate(WEEKDAYS))),
                                  categories=WEEKDAYS[:6], ordered=True)

by_wd = daily.groupby("weekday", observed=True).agg(days=("date", "size"),
                                                    total=("revenue", "sum"),
                                                    per_day=("revenue", "mean")).round(0)
by_wd

In [ ]:
pivot = cafe.pivot_table(index="weekday", columns="daypart", values="revenue",
                         aggfunc="sum", observed=True)
pivot_per_day = (pivot.div(by_wd["days"], axis=0) / 1000).round(1)     # 일평균 매출(천 원)
pivot_per_day

### ✏️ 직접 해보기 (Level 2)

1. 위 표를 `revenue` 대신 **판매 수량(`qty`)** 기준으로 바꿔 보세요. 가장 바쁜 칸이 바뀌나요?
2. `hour`별 일평균 주문 건수를 구해, 가장 붐비는 **한 시간**을 찾아보세요.

In [ ]:
hourly = cafe.groupby("hour")["order_id"].size() / cafe["date"].nunique()   # ← 수량 기준 등으로 바꿔 보세요
hourly.round(1)

### Q3. 시험 기간에 판매가 달라지는가?

In [ ]:
daily["exam_week"] = daily["date"].apply(lambda d: any(0 <= (d - s).days < 5 for s in EXAMS))
weekdays_only = daily[daily["weekday"] != "토"]            # 토요일은 영업 형태가 달라 제외
exam_cmp = weekdays_only.groupby("exam_week").agg(days=("date", "size"),
                                                  per_day_orders=("orders", "mean"),
                                                  per_day_revenue=("revenue", "mean")).round(0)
exam_cmp.index = ["평상시", "시험 주간"]
exam_cmp

> 시험 주간은 **10일**뿐입니다. 차이가 보여도 표본이 작아 우연일 수 있으니, 결론에 **"10일 기준"** 같은 단서를 붙입니다.

---
## 4. Level 3 적용하기: 두 데이터 합치기 – 기온과 판매

### Q4. 기온은 판매와 관련이 있는가?

In [ ]:
drinks = cafe[cafe["temp_type"].isin(["HOT", "ICE"])]
ice = drinks.pivot_table(index="date", columns="temp_type", values="qty", aggfunc="sum", fill_value=0)
ice["ice_ratio"] = ice["ICE"] / (ice["ICE"] + ice["HOT"])

day_df = (daily.merge(ice[["ice_ratio"]].reset_index(), on="date", how="left")
               .merge(weather[["date", "avg_temp", "precipitation_mm"]], on="date", how="left", validate="one_to_one"))
assert len(day_df) == len(daily)
day_df.head()

In [ ]:
print("평균기온 결측일:", day_df["avg_temp"].isna().sum())
print(day_df[["avg_temp", "ice_ratio", "revenue", "precipitation_mm"]].corr().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(day_df["avg_temp"], day_df["ice_ratio"] * 100, s=14, alpha=0.7)
axes[0].set_xlabel("평균기온 (℃)")
axes[0].set_ylabel("아이스 음료 비율 (%)")
axes[0].set_title("기온이 높을수록 아이스 비율 ↑")
axes[1].scatter(day_df["avg_temp"], day_df["revenue"] / 1000, s=14, alpha=0.7, color="#E45756")
axes[1].set_xlabel("평균기온 (℃)")
axes[1].set_ylabel("일 매출 (천 원)")
axes[1].set_title("기온과 일 매출은 관계가 약함")
plt.tight_layout()
plt.show()

In [ ]:
bands = pd.cut(day_df["avg_temp"], bins=[-np.inf, 10, 15, 20, 25, np.inf],
               labels=["10℃ 미만", "10~15", "15~20", "20~25", "25℃ 이상"])
day_df.groupby(bands, observed=True).agg(days=("date", "size"),
                                         ice_pct=("ice_ratio", "mean"),
                                         revenue=("revenue", "mean")).assign(ice_pct=lambda d: (d["ice_pct"] * 100).round(1),
                                                                             revenue=lambda d: d["revenue"].round(0))

### 발견 정리 (예시)

| 발견 | 근거 | 주의점 |
|---|---|---|
| 아메리카노·카페라떼가 매출의 40% 이상 (23.6% + 18.6%) | 메뉴별 매출 비중 | 한 학기(3~6월) 데이터 |
| 평일 점심(11~13시)이 가장 바쁨, 특히 12시 | 요일×시간대 일평균 매출, 시간별 주문 | 일요일 휴무, 토요일은 매출이 평일의 절반 이하 |
| 기온이 높을수록 아이스 음료 비율이 크게 오름 (10℃ 미만 26% → 25℃ 이상 81%) | 상관계수 0.85, 기온 구간별 비율 | **상관이지 인과는 아님**, 계절·시험 일정과 겹침 |
| 시험 주간 일평균 주문이 늘어남 (34건 → 41건) | 평상시 vs 시험 주간 (평일) | 시험 주간은 10일뿐 |

좋은 발견은 **숫자 + 비교 기준 + 한계**를 함께 씁니다.

---
## 5. Level 4 AI와 함께: 분석 해석 검증

AI에게 데이터를 주고 "분석해서 인사이트를 알려 줘"라고 하면 그럴듯한 문장을 받습니다. 다음 세 문장을 **데이터로 검증**해 봅시다.

> 🤖 "가장 인기 있는 메뉴는 아메리카노이며, 주문의 40% 이상을 차지합니다."
> 🤖 "3월 매출이 6월보다 훨씬 높으므로 학기 초에 수요가 몰립니다."
> 🤖 "기온이 오르면 매출이 증가하므로 여름에 메뉴 가격을 올리면 수익이 늘어납니다."

In [ ]:
# 검증 1: '주문의 40% 이상'?
share_rows = (cafe["menu"] == "아메리카노").mean() * 100
share_qty = cafe.loc[cafe["menu"] == "아메리카노", "qty"].sum() / cafe["qty"].sum() * 100
print(f"주문 건수 비중 {share_rows:.1f}% / 판매 수량 비중 {share_qty:.1f}%")

In [ ]:
# 검증 2: 3월 vs 6월 – 합계 vs 일평균
monthly = daily.groupby(daily["date"].dt.month).agg(days=("date", "size"), total=("revenue", "sum"), per_day=("revenue", "mean")).round(0)
monthly

In [ ]:
# 검증 3: 기온과 매출의 상관
print("기온-매출 상관:", round(day_df["avg_temp"].corr(day_df["revenue"]), 2))

| AI 문장 | 검증 결과 |
|---|---|
| 주문의 40% 이상 | 아메리카노 비중은 건수 **29.1%**, 수량 28.9%, 매출 23.6% → 과장. 기준(건수/수량/매출)도 밝히지 않음 |
| 3월 매출이 6월보다 훨씬 높다 | 합계는 3월이 크지만 6월은 **20일까지**만 있어 영업일이 17일. **일평균은 오히려 6월이 높음** |
| 기온이 오르면 매출 증가 → 가격 인상 | 기온-매출 상관은 **0.1로 약함**. 상관은 인과가 아니며, 가격 인상 효과는 이 데이터로 알 수 없음 |

### 📝 AI에게 분석을 맡길 때

```text
cafe_sales.csv의 한 행은 '주문 한 건의 메뉴 한 종류'이고 qty는 수량입니다.
- '인기'는 판매 수량 기준으로, 비율은 소수 첫째 자리까지 계산 코드와 함께 보여 주세요.
- 월·요일 비교는 영업일 수가 다르므로 일평균으로 비교하세요.
- 상관관계를 인과로 해석하지 말고, 각 주장마다 근거가 되는 표를 제시하세요.
```

AI의 **문장**이 아니라 **문장을 만든 계산**을 요구하고, 직접 실행해 확인합니다.

---
## 6. 최종 프로젝트 기획

### 최종 프로젝트 개요

- **주제**: 공개 데이터 또는 수업 데이터로 **연구 질문 2~3개**에 답하는 재현 가능한 분석
- **산출물**: 분석 노트북(Colab), README, 발표 자료(5분), AI 활용·성찰 보고서
- **일정**: 12주 제안서 → 13주 시각화 → 14주 테스트·리팩토링 → 15주 발표
- 자세한 기준: [최종 프로젝트 안내](../../projects/final-project.md)

### 좋은 연구 질문

| ❌ 막연한 질문 | ✅ 좋은 질문 |
|---|---|
| 카페 매출을 분석한다 | 평일 점심 시간대 매출은 다른 시간대보다 얼마나 높은가? |
| 날씨가 판매에 영향을 줄까? | 평균기온 구간별로 아이스 음료 판매 비율은 어떻게 달라지는가? |
| 학생들은 AI를 많이 쓸까? | AI 사용 여부에 따라 학년별 수업 만족도 평균에 차이가 있는가? |

**체크리스트**: 데이터로 답할 수 있는가? / 비교 대상이 있는가? / 결과를 숫자·표·그래프로 보여 줄 수 있는가? / 2주 안에 끝낼 수 있는가?

### 데이터 후보

| 데이터 | 할 수 있는 질문 예 |
|---|---|
| `cafe_sales.csv` + `weather_daily.csv` | 시간대·요일·기온·시험 기간별 판매 패턴 |
| `survey_raw.csv` (정제 후) + `departments.csv` | AI 사용과 코딩 시간·만족도의 관계 |
| `students.csv` | 출석·과제와 성적의 관계 (가상 데이터 한계 명시) |
| 공공데이터포털, 기상자료개방포털, KOSIS 등 | 직접 고른 주제 (출처·이용 조건 확인) |

In [ ]:
# 제안서 초안을 바로 파일로 저장해 둘 수 있습니다 (Colab에서는 왼쪽 파일 탭에서 내려받기)
proposal = """# 최종 프로젝트 제안서 (초안)

- 이름 / 학번:
- 제목:

## 1. 연구 질문 (2~3개)
1.
2.

## 2. 데이터
- 출처와 이용 조건:
- 한 행의 의미, 기간, 주요 열:

## 3. 분석 방법
- 정제:
- 비교·집계 방법:
- 예상 그래프 2~3개:

## 4. 예상 결과와 한계
-

## 5. AI 활용 계획
- 어디에 쓰고 어떻게 검증할 것인가:
"""
Path("proposal_draft.md").write_text(proposal, encoding="utf-8")
print(proposal)

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| EDA 절차 | 질문 → 데이터 이해 → 단변량 → 이변량·그룹 → 발견 정리 |
| 데이터 이해 | **한 행의 의미**, 기간, 집계 단위(건수·수량·매출) |
| 파생 변수 | `.dt.month`, `.dt.dayofweek`, `pd.cut`, 순서 있는 `Categorical` |
| 공정한 비교 | 그룹별 일수가 다르면 **일평균** |
| 두 데이터 | 날짜로 `merge(validate=...)`, 행 수 확인 |
| 해석 | 숫자 + 비교 기준 + 한계, **상관 ≠ 인과**, 작은 표본 주의 |
| AI 검증 | 문장이 아니라 계산을 요구하고 직접 실행 |
| 기획 | 답할 수 있고 비교가 있는 구체적 연구 질문 |

### 이번 주 할 일

- [ ] 과제 노트북 (문제 4에 **최종 프로젝트 제안서 초안** 포함)
- [ ] 프로젝트 데이터를 정하고 첫 EDA 실행해 보기

### 다음 주 예고

**Week 13 · matplotlib 데이터 시각화**: 오늘 찾은 발견을 **메시지가 분명한 차트**로 만듭니다.